In [ ]:
import pickle as pkl
import itertools
import os

import pandas as pd
import numpy as np
import scanpy as sc
import networkx as nx
from scipy import stats
import sklearn.metrics as metrics
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
#from matplotlib_venn import venn3
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

#from upsetplot import UpSet, from_memberships
from collections import Counter

plt.rcParams['figure.dpi'] = 300

In [ ]:
DPI = 300

methods=['celloracle']

SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+"/Wu_2021/CID4495" for method in methods}

SAMPLE = 'CID4495'#SAMPLE_PATHS['celloracle'].split('/')[-1].split('_')[0]

def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

clones = get_clones(SAMPLE_PATHS['celloracle'])

temp = []
for clone in clones:
    if clone.isdigit() or clone=='N':
        temp.append(clone)

clones = temp.copy()

clones

In [ ]:
boot_grns = {}

for method in methods:
    boot_grns = {x:{} for x in clones}
    for clone in clones:
        for rate in [0.9]:
            boot_grns[clone][rate] = []
            for i in range(1,101):
                network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                if method=='celloracle':
                    G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                if method=='grnboost':
                    thresh = np.histogram(network['importance'])[1][1]
                    G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                boot_grns[clone][rate].append(G)

In [ ]:
all_grns = {}

for method in methods:
    for clone in clones:
        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-whole-"+method+"-net.tsv", sep='\t')
        if method=='celloracle':
            G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
        if method=='grnboost':
            thresh = np.histogram(network['importance'])[1][1]
            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
        all_grns[clone] = G

In [ ]:
for method in methods:
    for clone in clones:
        for rate in [0.90]:
            for idx in range(100):
                _G = boot_grns[clone][rate][idx]
                for e in _G.edges():
                    if all_grns[clone].has_edge(e[0], e[1]):
                        if not f"support_{rate}" in all_grns[clone].edges[e[0], e[1]].keys():
                            all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] = 1
                        else:
                            all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] += 1

In [ ]:
G_data = {clone:{"source": [], "target": [], "support_0.90": []} for clone in clones}
for method in methods:
    for clone in clones:
        for s, t, vals in all_grns[clone].edges(data=True):
            G_data[clone]["source"].append(s)
            G_data[clone]["target"].append(t)
            try:
                G_data[clone]["support_0.90"].append(vals["support_0.9"])
            except KeyError:
                G_data[clone]["support_0.90"].append(0)

G_df = {clone:pd.DataFrame(data=G_data[clone]) for clone in clones}

In [ ]:
for method in methods:
    cl_edges = {}
    for clone in clones:
        unique_edges = set(all_grns[clone].edges())
        for clone2 in clones:
            if clone2 != clone:
                unique_edges = unique_edges - set(all_grns[clone2].edges())
        cl_edges[clone] = unique_edges
        G_df[clone]["edge"] = list(zip(G_df[clone]["source"], G_df[clone]["target"]))
        G_df[clone]['unique'] = [True if edge in cl_edges[clone] else False for edge in G_df[clone].edge]

In [ ]:
G_df_avg = {}
all_grns_combined = {}
for clone in clones:
    G_df_avg[clone] = pd.merge(G_df['grnboost'][clone], G_df['celloracle'][clone], how='inner', on=['source', 'target'])
    G_df_avg[clone]['support_0.90'] = G_df_avg[clone]['support_0.90_x']#(G_df_avg[clone]['support_0.90_x'] + G_df_avg[clone]['support_0.90_y']) / 2
    G_df_avg[clone] = G_df_avg[clone].drop(columns=['edge_x', 'edge_y', 'support_0.90_x', 'support_0.90_y', 'unique_x', 'unique_y'])
    all_grns_combined[clone] = nx.intersection(all_grns['celloracle'][clone], all_grns['grnboost'][clone])

In [ ]:
cl_edges = {}
for clone in clones:
    unique_edges = set(all_grns_combined[clone].edges())
    for clone2 in clones:
        if clone2 != clone:
            unique_edges = unique_edges - set(all_grns_combined[clone2].edges())
    cl_edges[clone] = unique_edges
    G_df_avg[clone]["edge"] = list(zip(G_df_avg[clone]["source"], G_df_avg[clone]["target"]))
    G_df_avg[clone]['unique'] = [True if edge in cl_edges[clone] else False for edge in G_df_avg[clone].edge]

In [ ]:
# Find genes present in all networks
all_nodes_per_group = [set(G.nodes()) for G in all_grns.values()]
common_nodes = set.intersection(*all_nodes_per_group)

# Initialize result container
similarity = {'successor':{}, 'predecessor':{}}  # Dict: gene -> dict of pairwise Jaccard similarities
genes_not_in_network = []

# Iterate over genes
for gene in common_nodes:
    pairwise_jaccard = {'successor':{}, 'predecessor':{}}

    for g1, g2 in itertools.combinations(clones, 2):
        G1, G2 = all_grns[g1], all_grns[g2]

        # Sanity check
        if gene not in G1.nodes or gene not in G2.nodes:
            genes_not_in_network.append((gene, g1, g2))
            continue
        
        #Succesor
        succ1 = set(G1.successors(gene))
        succ2 = set(G2.successors(gene))

        union = succ1 | succ2
        if len(union) == 0:
            jaccard = float('nan')
        else:
            jaccard = len(succ1 & succ2) / len(union)

        pairwise_jaccard['successor'][f"{g1}_vs_{g2}"] = jaccard
        
        #Predecessor
        pre1 = set(G1.predecessors(gene))
        pre2 = set(G2.predecessors(gene))

        union = pre1 | pre2
        if len(union) == 0:
            jaccard = float('nan')
        else:
            jaccard = len(pre1 & pre2) / len(union)

        pairwise_jaccard['predecessor'][f"{g1}_vs_{g2}"] = jaccard

    similarity['successor'][gene] = pairwise_jaccard['successor']
    similarity['predecessor'][gene] = pairwise_jaccard['predecessor']

# Convert to DataFrame for inspection or saving
similarity_succ_df = pd.DataFrame.from_dict(similarity['successor'], orient='index')
similarity_pre_df = pd.DataFrame.from_dict(similarity['predecessor'], orient='index')

In [ ]:
fig, axs = plt.subplots(1, len(clones)-1, figsize=(2*(len(clones)-1)*1.5,3*1.5), sharey=True)

fig.suptitle(SAMPLE, fontsize=16)

for idx in range(len(clones)-1):
    __90_df = G_df[[x for x in clones if x!='N'][idx]][["edge", "unique", "support_0.90"]].copy().rename(columns=lambda x: x.split("_")[0])
    __90_df["rate"] = 0.90
    c1_plot_df = __90_df.copy()
    ax = sns.violinplot(data=c1_plot_df, x="rate", y="support", hue="unique", 
                split=True, inner="quart", ax=axs[idx], legend=False, palette='tab10')
    ax.tick_params(left=True)

    axs[idx].set_ylim((0, 100))
    axs[idx].set_yticks(range(0, 101, 20))
    axs[idx].set_yticklabels(range(0, 101, 20), fontsize=16)
    axs[idx].set_ylabel("Bootstrap support", fontsize=18)

    axs[idx].set_title("Clone "+[x for x in clones if x!='N'][idx], fontsize=16)
    
    axs[idx].set_xticks([0])
    axs[idx].set_xticklabels([f"{rate:0.2f}" for rate in [0.9]], fontsize=12)
    axs[idx].set_xlabel("Subsampling fraction", fontsize=14)

plt.tight_layout()
#fig.savefig(f"{SAMPLE_PATH}/bootstrap-edge-supp-{SAMPLE}-dpi{DPI}.png", dpi=DPI)
plt.show()

In [ ]:
edge_sets = [all_grns[clone].edges for clone in clones]

# Create a mapping from each edge to the groups it belongs to
edge_to_groups = {}
for group_name, edge_set in zip(clones, edge_sets):
    for edge in edge_set:
        edge_to_groups.setdefault(edge, set()).add(group_name)

# Convert to list of group memberships
memberships = [tuple(sorted(group_list)) for group_list in edge_to_groups.values()]

# Create data for UpSetPlot
data = from_memberships(memberships)

# Plot
fig = plt.figure(figsize=(12, 6))
UpSet(data, subset_size='count', show_counts=True).plot(fig=fig)

plt.suptitle("GRN Edge Overlaps", fontsize=16)
plt.tight_layout()
#fig.savefig(f"{SAMPLE_PATH}/upset-edges-{SAMPLE}-dpi{DPI}.png", dpi=DPI)
plt.show()

In [ ]:
count_clone = 0
count_norm = 0
for key,val in dict(Counter(memberships)).items():
    if len(key)>1 and len(key)<len(clones) and 'N' in key:
        count_norm+=val
    elif len(key)>1 and len(key)<len(clones) and 'N' not in key:
        count_clone+=val

print(count_clone, count_norm)


In [ ]:
#JUNB,JUN  JUNB,ID3  MYC,TRIB1  MYC,CD14
G_df_avg['2'].loc[G_df_avg['2']['edge']==('MYC', 'CD14')]['support_0.90']

In [ ]:
Counter(memberships)

In [ ]:
for col in similarity_succ_df.columns:
    values = similarity_succ_df[col].dropna().values  # Remove NaNs

    if len(values) == 0:
        continue  # Skip empty columns

    mu = np.mean(values)
    std = np.std(values)

    plt.figure(figsize=(5.5, 4))
    sns.histplot(values, legend=False, stat="proportion", bins=30)
    plt.axvline(mu-3*std, linestyle="--", linewidth=2, color='#EA3C77')

    plt.text(0.1, 0.075, r"$\mu=$" + f"{mu:.4f}", fontsize=12)
    plt.text(0.1, 0.0625, r"$\sigma=$" + f"{std:.4f}", fontsize=12)

    plt.title(f"{col} - {SAMPLE}", fontsize=16)
    plt.xlabel("Jaccard similarity (successor)", fontsize=14)
    plt.ylabel("Fraction", fontsize=14)

    plt.xlim((0., 1.))
    plt.xticks(np.arange(0., 1.01, 0.2), fontsize=12)
    # Uncomment and adjust if needed
    # plt.ylim((0., 0.25))
    # plt.yticks(np.arange(0., 0.26, 0.05), fontsize=12)

    plt.tight_layout()
    # Uncomment to save:
    # plt.savefig(f"plots/{SAMPLE}/hist_{col}_dpi{DPI}.png", dpi=DPI)
    plt.show()

In [ ]:
for col in similarity_pre_df.columns:
    values = similarity_pre_df[col].dropna().values  # Remove NaNs

    if len(values) == 0:
        continue  # Skip empty columns

    mu = np.mean(values)
    std = np.std(values)

    plt.figure(figsize=(5.5, 4))
    sns.histplot(values, legend=False, stat="proportion", bins=30)
    plt.axvline(mu-3*std, linestyle="--", linewidth=2, color='#EA3C77')

    plt.text(0.1, 0.075, r"$\mu=$" + f"{mu:.4f}", fontsize=12)
    plt.text(0.1, 0.0625, r"$\sigma=$" + f"{std:.4f}", fontsize=12)

    plt.title(f"{col} - {SAMPLE}", fontsize=16)
    plt.xlabel("Jaccard similarity (predecessor)", fontsize=14)
    plt.ylabel("Fraction", fontsize=14)

    plt.xlim((0., 1.))
    plt.xticks(np.arange(0., 1.01, 0.2), fontsize=12)
    # Uncomment and adjust if needed
    # plt.ylim((0., 0.25))
    # plt.yticks(np.arange(0., 0.26, 0.05), fontsize=12)

    plt.tight_layout()
    # Uncomment to save:
    # plt.savefig(f"plots/{SAMPLE}/hist_{col}_dpi{DPI}.png", dpi=DPI)
    plt.show()

## Summary plots

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}

total_G_df = {}
total_avg_G_df = {}
total_all_grns = {}
total_all_grns_combined = {}

for paper, datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['grnboost','celloracle']

        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = SAMPLE_PATHS['celloracle'].split('/')[-1]
        print('sample',SAMPLE)

        clones = get_clones(SAMPLE_PATHS['celloracle'])

        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()
        print('clones',clones)

        boot_grns = {x:{} for x in methods}

        for method in methods:
            boot_grns[method] = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[method][clone][rate] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                        if method=='celloracle':
                            G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        if method=='grnboost':
                            thresh = np.histogram(network['importance'])[1][1]
                            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        boot_grns[method][clone][rate].append(G)

        all_grns = {x:{} for x in methods}

        for method in methods:
            for clone in clones:
                network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-whole-"+method+"-net.tsv", sep='\t')
                if method=='celloracle':
                    G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                if method=='grnboost':
                    thresh = np.histogram(network['importance'])[1][1]
                    G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                all_grns[method][clone] = G
                
        for method in methods:
            for clone in clones:
                for rate in [0.90]:
                    for idx in range(100):
                        _G = boot_grns[method][clone][rate][idx]
                        for e in _G.edges():
                            if all_grns[method][clone].has_edge(e[0], e[1]):
                                if not f"support_{rate}" in all_grns[method][clone].edges[e[0], e[1]].keys():
                                    all_grns[method][clone].edges[e[0], e[1]][f"support_{rate}"] = 1
                                else:
                                    all_grns[method][clone].edges[e[0], e[1]][f"support_{rate}"] += 1

        G_data = {method:{clone:{"source": [], "target": [], "support_0.90": []} for clone in clones} for method in methods}
        for method in methods:
            for clone in clones:
                for s, t, vals in all_grns[method][clone].edges(data=True):
                    G_data[method][clone]["source"].append(s)
                    G_data[method][clone]["target"].append(t)
                    try:
                        G_data[method][clone]["support_0.90"].append(vals["support_0.9"])
                    except KeyError:
                        G_data[method][clone]["support_0.90"].append(0)

        G_df = {method:{clone:pd.DataFrame(data=G_data[method][clone]) for clone in clones} for method in methods}

        for method in methods:
            cl_edges = {}
            for clone in clones:
                unique_edges = set(all_grns[method][clone].edges())
                for clone2 in clones:
                    if clone2 != clone:
                        unique_edges = unique_edges - set(all_grns[method][clone2].edges())
                cl_edges[clone] = unique_edges
                G_df[method][clone]["edge"] = list(zip(G_df[method][clone]["source"], G_df[method][clone]["target"]))
                G_df[method][clone]['unique'] = [True if edge in cl_edges[clone] else False for edge in G_df[method][clone].edge]

        G_df_avg = {}
        all_grns_combined = {}
        for clone in clones:
            G_df_avg[clone] = pd.merge(G_df['grnboost'][clone], G_df['celloracle'][clone], how='inner', on=['source', 'target'])
            G_df_avg[clone]['support_0.90'] = (G_df_avg[clone]['support_0.90_x'] + G_df_avg[clone]['support_0.90_y']) / 2
            G_df_avg[clone] = G_df_avg[clone].drop(columns=['edge_x', 'edge_y', 'support_0.90_x', 'support_0.90_y', 'unique_x', 'unique_y'])
            all_grns_combined[clone] = nx.intersection(all_grns['celloracle'][clone], all_grns['grnboost'][clone])
        
        cl_edges = {}
        for clone in clones:
            unique_edges = set(all_grns_combined[clone].edges())
            for clone2 in clones:
                if clone2 != clone:
                    unique_edges = unique_edges - set(all_grns_combined[clone2].edges())
            cl_edges[clone] = unique_edges
            G_df_avg[clone]["edge"] = list(zip(G_df_avg[clone]["source"], G_df_avg[clone]["target"]))
            G_df_avg[clone]['unique'] = [True if edge in cl_edges[clone] else False for edge in G_df_avg[clone].edge]

        total_G_df[dataset] = G_df.copy()
        total_avg_G_df[dataset] = G_df_avg.copy()
        total_all_grns[dataset] = all_grns.copy()
        total_all_grns_combined[dataset] = all_grns_combined.copy()

pkl.dump(total_G_df, open('/shared/nr57/GRN/data/3CA/total_G_df_new.pkl','wb'))
pkl.dump(total_avg_G_df, open('/shared/nr57/GRN/data/3CA/total_avg_G_df_new.pkl','wb'))
pkl.dump(total_all_grns, open('/shared/nr57/GRN/data/3CA/total_all_grns_new.pkl','wb'))
pkl.dump(total_all_grns_combined, open('/shared/nr57/GRN/data/3CA/total_all_grns_combined_new.pkl','wb'))

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

#datasets = ['/BC/CID4513', '/BC/CID4515', '/BC/CID4523', '/BC/CID44971', '/BC/CID44991', '/CRC/SMC01', '/CRC/SMC04', '/CRC/SMC07', '/CRC/SMC14', '/CRC/SMC18', '/CRC/SMC23', '/CRC/SMC25']
all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}

total_G_df = {}
total_avg_G_df = {}
total_all_grns = {}
total_all_grns_combined = {}

for paper,datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['grnboost','celloracle']

        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = SAMPLE_PATHS['celloracle'].split('/')[-1]
        print('sample',SAMPLE)

        clones = get_clones(SAMPLE_PATHS['celloracle'])
        print('clones',clones)

        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()

        print(clones)

        boot_grns = {x:{} for x in methods}

        for method in methods:
            boot_grns[method] = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[method][clone][rate] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                        if method=='celloracle':
                            G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        if method=='grnboost':
                            thresh = np.histogram(network['importance'])[1][1]
                            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        boot_grns[method][clone][rate].append(G)

        all_grns = {x:{} for x in methods}

        for method in methods:
            for clone in clones:
                network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-whole-"+method+"-net.tsv", sep='\t')
                if method=='celloracle':
                    G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                if method=='grnboost':
                    thresh = np.histogram(network['importance'])[1][1]
                    G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                all_grns[method][clone] = G
        
        all_grns = {clone:nx.intersection(all_grns['celloracle'][clone], all_grns['grnboost'][clone]) for clone in clones}
        
        for clone in clones:
            for rate in [0.90]:
                for idx in range(100):
                    _G = nx.intersection(boot_grns['celloracle'][clone][rate][idx], boot_grns['grnboost'][clone][rate][idx])
                    for e in _G.edges():
                        if all_grns[clone].has_edge(e[0], e[1]):
                            if not f"support_{rate}" in all_grns[clone].edges[e[0], e[1]].keys():
                                all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] = 1
                            else:
                                all_grns[clone].edges[e[0], e[1]][f"support_{rate}"] += 1

        G_data = {clone:{"source": [], "target": [], "support_0.90": []} for clone in clones}
        for clone in clones:
            for s, t, vals in all_grns[clone].edges(data=True):
                G_data[clone]["source"].append(s)
                G_data[clone]["target"].append(t)
                try:
                    G_data[clone]["support_0.90"].append(vals["support_0.9"])
                except KeyError:
                    G_data[clone]["support_0.90"].append(0)

        G_df = {clone:pd.DataFrame(data=G_data[clone]) for clone in clones}

        cl_edges = {}
        for clone in clones:
            unique_edges = set(all_grns[clone].edges())
            for clone2 in clones:
                if clone2 != clone:
                    unique_edges = unique_edges - set(all_grns[clone2].edges())
            cl_edges[clone] = unique_edges
            G_df[clone]["edge"] = list(zip(G_df[clone]["source"], G_df[clone]["target"]))
            G_df[clone]['unique'] = [True if edge in cl_edges[clone] else False for edge in G_df[clone].edge]

        total_G_df[dataset] = G_df.copy()
        total_all_grns[dataset] = all_grns.copy()


pkl.dump(total_G_df, open('/shared/nr57/GRN/data/3CA/consensus_G_df_new.pkl','wb'))
pkl.dump(total_all_grns, open('/shared/nr57/GRN/data/3CA/consensus_all_grns_new.pkl','wb'))

In [ ]:
total_G_df = pkl.load(open('/shared/nr57/GRN/data/3CA/total_G_df_new.pkl','rb'))
total_avg_G_df = pkl.load(open('/shared/nr57/GRN/data/3CA/total_avg_G_df_new.pkl','rb'))
total_all_grns = pkl.load(open('/shared/nr57/GRN/data/3CA/total_all_grns_new.pkl','rb'))
total_all_grns_combined = pkl.load(open('/shared/nr57/GRN/data/3CA/total_all_grns_combined_new.pkl','rb'))
consensus_G_df = pkl.load(open('/shared/nr57/GRN/data/3CA/consensus_G_df_new.pkl','rb'))
consensus_all_grns = pkl.load(open('/shared/nr57/GRN/data/3CA/consensus_all_grns_new.pkl','rb'))

In [ ]:
nx.to_pandas_edgelist(total_all_grns['CID4495']['celloracle']['0'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_avg_edges(samples_dict):
    """
    samples_dict structure:
    {
        sample1: {
            method1: {cat1: G, cat2: G, ...},
            method2: {...},
        },
        sample2: {...},
        ...
    }
    """

    samples = list(samples_dict.keys())

    # Collect all methods across all samples
    all_methods = set()
    for sample in samples:
        all_methods.update(samples_dict[sample].keys())
    all_methods = sorted(list(all_methods))

    # Compute average edge counts
    avg_edge_counts = {method: [] for method in all_methods}

    for sample in samples:
        methods = samples_dict[sample]

        for method in all_methods:
            if method not in methods:
                avg_edge_counts[method].append(np.nan)
                continue

            cat_graphs = methods[method]

            # Compute average edges across categories
            n_edges = [G.number_of_edges() for G in cat_graphs.values()]
            avg_edge_counts[method].append(np.mean(n_edges))

    # --- Plotting ---
    x = np.arange(len(samples))  # sample positions
    width = 0.8 / len(all_methods)  # bar width per method

    fig, ax = plt.subplots(figsize=(18, 6))

    # Plot each method with bar offsets
    for i, method in enumerate(all_methods):
        offsets = x + i * width - (0.8 - width) / 2
        values = avg_edge_counts[method]

        bars = ax.bar(offsets, values, width, label=method)

        # --- Add value labels above bars ---
        for bar, val in zip(bars, values):
            if not np.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width()/2,
                    bar.get_height(),
                    f"{val:.1f}",
                    ha="center",
                    va="bottom",
                    fontsize=10
                )

    ax.set_xticks(x)
    ax.set_xticklabels(samples, fontsize=12)
    ax.set_ylabel("Average Number of Edges", fontsize=14)
    ax.set_xlabel("Samples", fontsize=14)
    ax.set_title("Average Edge Count per Method Across Categories", fontsize=16)
    ax.legend(title="Method", title_fontsize=12, fontsize=10)

    plt.tight_layout()
    plt.show()

temp_dict = total_all_grns.copy()
for dataset in datasets:
    temp_dict[dataset]['consensus'] = consensus_all_grns[dataset].copy()

plot_avg_edges(temp_dict)

In [ ]:
import math

def plot_avg_unique_edges(data_dict):
    """
    data_dict format:
    {
        'sample1': {
            'method1': { 'cat1': df, 'cat2': df, ... },
            'method2': { 'cat1': df, 'cat2': df, ... },
        },
        'sample2': { ... }
    }
    """

    samples = list(data_dict.keys())

    # Collect all methods across all samples
    all_methods = sorted({
        method
        for sample in samples
        for method in data_dict[sample].keys()
    })

    # Compute average unique edge count
    avg_unique_counts = {method: [] for method in all_methods}

    for sample in samples:
        sample_methods = data_dict[sample]

        for method in all_methods:
            if method not in sample_methods:
                avg_unique_counts[method].append(np.nan)
                continue

            cat_dfs = sample_methods[method]

            # Count unique edges per category: unique == True
            unique_counts = []
            for df in cat_dfs.values():
                count = df[df["unique"] == True].shape[0]
                unique_counts.append(count)

            avg_unique_counts[method].append(np.mean(unique_counts))

    # --- Plotting ---
    x = np.arange(len(samples))  # sample positions
    width = 0.8 / len(all_methods)  # bar width per method

    fig, ax = plt.subplots(figsize=(18, 6))

    for i, method in enumerate(all_methods):
        offsets = x + i * width - (0.8 - width) / 2
        values = avg_unique_counts[method]

        bars = ax.bar(offsets, values, width, label=method)

        # Add text labels
        for bar, val in zip(bars, values):
            if not math.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    f"{val:.1f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(samples, fontsize=12)
    ax.set_ylabel("Average # Unique Edges", fontsize=14)
    ax.set_xlabel("Samples", fontsize=14)
    ax.set_title("Average Unique Edges Per Method Across Clone", fontsize=16)
    ax.legend(title="Method", title_fontsize=12, fontsize=10)

    plt.tight_layout()
    plt.show()

temp_dict = total_G_df.copy()
for dataset in datasets:
    temp_dict[dataset]['consensus'] = consensus_G_df[dataset].copy()

plot_avg_unique_edges(temp_dict)

In [ ]:
import math

def plot_avg_nodes(data_dict):
    """
    data_dict format:
    {
        'sample1': {
            'method1': { 'cat1': nx_graph, 'cat2': nx_graph, ... },
            'method2': { 'cat1': nx_graph, 'cat2': nx_graph, ... },
        },
        'sample2': { ... }
    }
    """

    samples = list(data_dict.keys())

    # Collect all methods across all samples
    all_methods = sorted({
        method
        for sample in samples
        for method in data_dict[sample].keys()
    })

    # Compute avg node count
    avg_node_counts = {method: [] for method in all_methods}

    for sample in samples:
        sample_methods = data_dict[sample]

        for method in all_methods:
            if method not in sample_methods:
                # Missing method for this sample → record NaN
                avg_node_counts[method].append(np.nan)
                continue

            cat_graphs = sample_methods[method]

            # Count nodes in every category graph
            node_counts = [g.number_of_nodes() for g in cat_graphs.values()]

            avg_node_counts[method].append(np.mean(node_counts))

    # --- Plotting ---
    x = np.arange(len(samples))  # sample positions
    width = 0.8 / len(all_methods)  # width per bar

    fig, ax = plt.subplots(figsize=(18, 6))

    for i, method in enumerate(all_methods):
        offsets = x + i * width - (0.8 - width) / 2
        values = avg_node_counts[method]

        bars = ax.bar(offsets, values, width, label=method)

        # Add text labels
        for bar, val in zip(bars, values):
            if not math.isnan(val):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    f"{val:.1f}",
                    ha="center",
                    va="bottom",
                    fontsize=10,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(samples, fontsize=12)
    ax.set_ylabel("Average # Nodes", fontsize=14)
    ax.set_xlabel("Samples", fontsize=14)
    ax.set_title("Average Number of Nodes per Method Across Clones", fontsize=16)
    ax.legend(title="Method", title_fontsize=12, fontsize=10)

    plt.tight_layout()
    plt.show()

temp_dict = total_all_grns.copy()
for dataset in datasets:
    temp_dict[dataset]['consensus'] = consensus_all_grns[dataset].copy()
    for method in ['celloracle','grnboost_filtered','consensus']:
        for key in temp_dict[dataset]['consensus'].keys():
            temp_dict[dataset][method][key].remove_nodes_from(list(nx.isolates(temp_dict[dataset][method][key])))

plot_avg_nodes(temp_dict)

In [ ]:
for i in temp_dict['/BC/CID44971']['consensus'].keys():
    print(i)
    non_isolates = temp_dict['/BC/CID44971']['consensus'][i].nodes
    for node in non_isolates:
        # collect all edge weights to/from node
        weights = []
        for u, v, data in temp_dict['/BC/CID44971']['consensus'][i].edges(node, data=True):  # edges incident on 'node'
            w = data.get('support_0.9', None)
            if w is not None:
                weights.append(w)

        if len(weights) == 0:
            continue

        mean_w = np.mean(weights)
        std_w  = np.std(weights)

        print(f"Node: {node:<15} | Mean Weight: {mean_w:.4f} | Std Dev: {std_w:.4f}")

In [ ]:
from scipy.stats import wasserstein_distance
from scipy.stats import mannwhitneyu
from scipy.stats import kstest

#methods = ['grnboost', 'celloracle']
methods = ['celloracle']

distances = {dataset:{method:[] for method in methods} for dataset in total_avg_G_df.keys()}
#distances['consensus'] = []
ks_distances = {dataset:{method:[] for method in methods} for dataset in total_avg_G_df.keys()}
bootstraps = {'TNBC':{'non-unique':[], 'unique':[]}, 'CRC':{'non-unique':[], 'unique':[]}}
bootstraps_samples = {dataset:{'non-unique':[], 'unique':[]} for dataset in total_avg_G_df.keys()}
#ks_distances['consensus'] = []

for dataset in total_avg_G_df.keys():
    clones = total_avg_G_df[dataset].keys()
    for clone in clones:
        if clone != 'N':
            if len(consensus_G_df[dataset][clone]['unique'].unique())==2:
                #distances['consensus'].append(wasserstein_distance(consensus_G_df[dataset][clone].loc[consensus_G_df[dataset][clone]['unique']]['support_0.90'], consensus_G_df[dataset][clone].loc[~consensus_G_df[dataset][clone]['unique']]['support_0.90']))
                #ks_distances['consensus'].append(kstest(rvs=consensus_G_df[dataset][clone].loc[consensus_G_df[dataset][clone]['unique']]['support_0.90'], cdf=consensus_G_df[dataset][clone].loc[~consensus_G_df[dataset][clone]['unique']]['support_0.90']).pvalue)
                for method in methods:
                    if 'Triple' in dataset or 'CID' in dataset:
                        bootstraps['TNBC']['unique']+=total_G_df[dataset][method][clone].loc[total_G_df[dataset][method][clone]['unique']]['support_0.90'].tolist()
                        bootstraps['TNBC']['non-unique']+=total_G_df[dataset][method][clone].loc[~total_G_df[dataset][method][clone]['unique']]['support_0.90'].tolist()
                    if 'SMC' in dataset:
                        bootstraps['CRC']['unique']+=total_G_df[dataset][method][clone].loc[total_G_df[dataset][method][clone]['unique']]['support_0.90'].tolist()
                        bootstraps['CRC']['non-unique']+=total_G_df[dataset][method][clone].loc[~total_G_df[dataset][method][clone]['unique']]['support_0.90'].tolist()
                    bootstraps_samples[dataset]['unique']+=total_G_df[dataset][method][clone].loc[total_G_df[dataset][method][clone]['unique']]['support_0.90'].tolist()
                    bootstraps_samples[dataset]['non-unique']+=total_G_df[dataset][method][clone].loc[~total_G_df[dataset][method][clone]['unique']]['support_0.90'].tolist()
                    print(dataset,clone,len(total_G_df[dataset][method][clone].loc[total_G_df[dataset][method][clone]['unique']]['support_0.90']), len(total_G_df[dataset][method][clone].loc[~total_G_df[dataset][method][clone]['unique']]['support_0.90']))
                    distances[dataset][method].append(wasserstein_distance(total_G_df[dataset][method][clone].loc[total_G_df[dataset][method][clone]['unique']]['support_0.90'], total_G_df[dataset][method][clone].loc[~total_G_df[dataset][method][clone]['unique']]['support_0.90']))
                    #ks_distances[dataset][method].append(kstest(rvs=total_G_df[dataset][method][clone].loc[total_G_df[dataset][method][clone]['unique']]['support_0.90'], cdf=total_G_df[dataset][method][clone].loc[~total_G_df[dataset][method][clone]['unique']]['support_0.90']).pvalue)
                    ks_distances[dataset][method].append(mannwhitneyu(total_G_df[dataset][method][clone].loc[total_G_df[dataset][method][clone]['unique']]['support_0.90'], total_G_df[dataset][method][clone].loc[~total_G_df[dataset][method][clone]['unique']]['support_0.90'], alternative='less').pvalue)
            else:
                print(dataset,clone,'has only True or False',len(consensus_G_df[dataset][clone]['unique'].unique()))

In [ ]:
import random

mannwhitneyu(random.sample(bootstraps['TNBC']['unique'], int(len(bootstraps['TNBC']['unique']) * 0.05)), random.sample(bootstraps['TNBC']['non-unique'], int(len(bootstraps['TNBC']['unique']) * 0.05)), alternative='less').pvalue

In [ ]:
def edge_set(G):
    """Return edge set (as tuples) from a NetworkX graph."""
    return set(G.edges())

def compute_grn_edge_stats(total_all_grns, method):
    """
    Parameters
    ----------
    total_all_grns : dict
        Format: total_all_grns[sample][method][clone] = nx.Graph
    method : str
        GRN inference method key

    Returns
    -------
    results : dict
        {
            'per_sample': {
                sample: {
                    'common_edges_mean': float,
                    'unique_edges_mean': float,
                    'clone_vs_N_common_mean': float,
                    'clone_vs_N_unique_mean': float
                }
            },
            'overall': {
                'common_edges_mean': float,
                'unique_edges_mean': float,
                'clone_vs_N_common_mean': float,
                'clone_vs_N_unique_mean': float
            }
        }
    """

    per_sample_results = {}

    all_common = []
    all_unique = []
    all_cloneN_common = []
    all_cloneN_unique = []

    for sample in total_all_grns:
        grns = total_all_grns[sample][method]

        clones = [c for c in grns if c != 'N']
        if len(clones) < 2:
            continue

        # --- Collect edge sets ---
        clone_edge_sets = {c: edge_set(grns[c])-edge_set(grns['N']) for c in clones}

        # --- Common edges across ALL clones ---
        common_edges = set.intersection(*clone_edge_sets.values())
        common_count = len(common_edges)

        # --- Unique edges per clone ---
        unique_counts = []
        for c in clones:
            others = set.union(*[clone_edge_sets[o] for o in clones if o != c])
            unique_edges = clone_edge_sets[c] - others
            unique_counts.append(len(unique_edges))

        unique_mean = np.mean(unique_counts)

        # --- Clone vs N ---
        if 'N' in grns:
            N_edges = edge_set(grns['N'])

            cloneN_common = []
            cloneN_unique = []

            for c in clones:
                c_edges = edge_set(grns[c])

                common = c_edges & N_edges
                unique = c_edges - N_edges

                cloneN_common.append(len(common))
                cloneN_unique.append(len(unique))

            cloneN_common_mean = np.mean(cloneN_common)
            cloneN_unique_mean = np.mean(cloneN_unique)

        else:
            cloneN_common_mean = np.nan
            cloneN_unique_mean = np.nan

        # --- Store per sample ---
        per_sample_results[sample] = {
            'common_edges_mean': common_count,
            'unique_edges_mean': unique_mean,
            'clone_vs_N_common_mean': cloneN_common_mean,
            'clone_vs_N_unique_mean': cloneN_unique_mean
        }

        # --- Collect for overall ---
        all_common.append(common_count)
        all_unique.append(unique_mean)
        all_cloneN_common.append(cloneN_common_mean)
        all_cloneN_unique.append(cloneN_unique_mean)

    # --- Overall means ---
    overall_results = {
        'common_edges_mean': np.nanmean(all_common),
        'unique_edges_mean': np.nanmean(all_unique),
        'clone_vs_N_common_mean': np.nanmean(all_cloneN_common),
        'clone_vs_N_unique_mean': np.nanmean(all_cloneN_unique)
    }

    return {
        'per_sample': per_sample_results,
        'overall': overall_results
    }

compute_grn_edge_stats(total_all_grns, 'celloracle')

In [ ]:
sns.set_theme(font_scale=1.5)
sns.set_style("white")


def ks_boxplot(distances):
    # Convert dict to long-format DataFrame
    df = pd.DataFrame([
        {"method": m, "value": v}
        for m, vals in distances.items()
        for v in vals
    ])

    plt.figure(figsize=(5, 7))
    print(df)
    ax = sns.boxplot(x="method", y="value", data=df, palette='tab10')
    ax.tick_params(left=True)

    # Compute and annotate box stats for each method
    grouped = df.groupby("method")["value"]
    global_min, global_max = df["value"].min(), df["value"].max()

    for i, method in enumerate(distances.keys()):
        vals = grouped.get_group(method)
        print(method)

        q1 = vals.quantile(0.25)
        median = vals.quantile(0.50)
        q3 = vals.quantile(0.75)
        #print(q1,median,q3)

        # Position text slightly above Q3
        y = q3 + -0.8 * (global_max - global_min)

        ax.text(i, y,
                f"Q1={q1:.2e}\nMed={median:.2e}\nQ3={q3:.2e}",
                ha="center", va="bottom", fontsize=9)

    ax.set_title("Mann Whitney U test of unique support < non-unique support by Method")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("p-values")
    plt.tight_layout()
    plt.show()


sns.boxplot({'TNBC':[x for k,v in distances.items() for x in v['celloracle'] if ('CID' in k) or ('Triple' in k)], 'CRC':[x for k,v in distances.items() for x in v['celloracle'] if 'SMC' in k]}
)
plt.title('Distance between distribution of bootstrap values of non-unique and unique edges', fontsize=16, pad=20)
plt.xlabel('Dataset')
plt.ylabel('Wasserstein distance')
plt.show()

data_list = []
for label, d in [('TNBC', bootstraps['TNBC']), ('CRC', bootstraps['CRC'])]:
    for key, values in d.items():
        for val in values:
            data_list.append({'Category': key, 'Value': val, 'Source': label})

df = pd.DataFrame(data_list)

# Plot using the 'hue' parameter for side-by-side boxes
sns.boxplot(data=df, x='Source', y='Value', hue='Category', palette='tab10', showfliers=False)
plt.xlabel('Cancer type')
plt.ylabel('Bootstrap support')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.show()

#sample level
data_list = []
for label, d in [(x.replace('BRCA1_','').replace('Triple_negative_', 'TN'), bootstraps_samples[x]) for x in bootstraps_samples.keys() if ('CID' in x) or ('Triple' in x)]:
    for key, values in d.items():
        for val in values:
            data_list.append({'Category': key, 'Value': val, 'Source': label})

df = pd.DataFrame(data_list)

# Plot using the 'hue' parameter for side-by-side boxes
sns.boxplot(data=df, x='Source', y='Value', hue='Category', palette='tab10', showfliers=False)
plt.xlabel('Cancer type')
plt.xticks(rotation=90)
plt.ylabel('Bootstrap support')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.show()

data_list = []
for label, d in [(x, bootstraps_samples[x]) for x in bootstraps_samples.keys() if ('SMC' in x)]:
    for key, values in d.items():
        for val in values:
            data_list.append({'Category': key, 'Value': val, 'Source': label})

df = pd.DataFrame(data_list)

# Plot using the 'hue' parameter for side-by-side boxes
sns.boxplot(data=df, x='Source', y='Value', hue='Category', palette='tab10', showfliers=False)
plt.xlabel('Cancer type')
plt.xticks(rotation=90)
plt.ylabel('Bootstrap support')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.show()

ks_boxplot({'TNBC':[x for k,v in ks_distances.items() for x in v['celloracle'] if ('CID' in k or 'Triple' in k)], 'CRC':[x for k,v in ks_distances.items() for x in v['celloracle'] if 'SMC' in k]})

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

#datasets = ['/BC/CID4513', '/BC/CID4515', '/BC/CID4523', '/BC/CID44971', '/BC/CID44991', '/CRC/SMC01', '/CRC/SMC04', '/CRC/SMC07', '/CRC/SMC14', '/CRC/SMC18', '/CRC/SMC23', '/CRC/SMC25']
all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}

total_all_grns = {}
consensus_all_grns = {}

for paper, datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['celloracle']

        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = dataset

        clones = get_clones(SAMPLE_PATHS['celloracle'])

        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()

        print(clones)

        all_grns = {x:{} for x in methods}

        for method in methods:
            for clone in clones:
                network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-whole-"+method+"-net.tsv", sep='\t')
                if method=='celloracle':
                    G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                if method=='grnboost':
                    thresh = np.histogram(network['importance'])[1][1]
                    G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                all_grns[method][clone] = G
        
        
        all_grns_methods = all_grns.copy()
        all_grns = {clone:all_grns['celloracle'][clone] for clone in clones}#{clone:nx.intersection(all_grns['celloracle'][clone], all_grns['grnboost'][clone]) for clone in clones}
        
        total_all_grns[dataset] = all_grns_methods.copy()
        consensus_all_grns[dataset] = all_grns.copy()


In [ ]:
all(x in list(dict(Counter(memberships)).keys())[0] for x in ('0', '1'))

In [ ]:
combined_setvals = {'clone v clone':[], 'normal v clone':[]}

for dataset in consensus_all_grns.keys():
    print(dataset)
    clones = list(consensus_all_grns[dataset].keys())
    edge_sets = [consensus_all_grns[dataset][clone].edges for clone in clones]

    # Create a mapping from each edge to the groups it belongs to
    edge_to_groups = {}
    for group_name, edge_set in zip(clones, edge_sets):
        for edge in edge_set:
            edge_to_groups.setdefault(edge, set()).add(group_name)

    # Convert to list of group memberships
    memberships = [tuple(sorted(group_list)) for group_list in edge_to_groups.values()]

    clone_sum = 0
    norm_sum = 0
    count_clone = 0
    count_norm = 0
    for key,val in dict(Counter(memberships)).items():
        if len(key)>1 and len(key)<len(clones) and 'N' in key:
            count_norm+=1
            norm_sum+=val
        elif len(key)>1 and len(key)<len(clones) and 'N' not in key:
            count_clone+=1
            clone_sum+=val

    if clone_sum!=0 or count_clone!=0:
        combined_setvals['clone v clone'].append(clone_sum/count_clone)
    else:
        combined_setvals['clone v clone'].append(0)
    if norm_sum!=0 or count_norm!=0:
        combined_setvals['normal v clone'].append(norm_sum/count_norm)
    else:
        combined_setvals['normal v clone'].append(0)
    print(clone_sum, norm_sum)
    #data = from_memberships(memberships)

    # Plot
    #fig = plt.figure(figsize=(12, 6))
    #UpSet(data, subset_size='count', show_counts=True).plot(fig=fig)

    #plt.suptitle("GRN Edge Overlaps", fontsize=16)
    #plt.tight_layout()
    #fig.savefig(f"{SAMPLE_PATH}/upset-edges-{SAMPLE}-dpi{DPI}.png", dpi=DPI)
    #plt.show()

method_setvals = {'clone v clone':[], 'normal v clone':[]}
method_specific_setvals = {method:{'clone v clone':[], 'normal v clone':[]} for method in methods}
dataset_specific_setvals = {dataset:{'clone v clone':[], 'normal v clone':[]} for dataset in consensus_all_grns.keys()}

for dataset in consensus_all_grns.keys():
    print(dataset)
    for method in methods:
        clones = list(total_all_grns[dataset][method].keys())
        edge_sets = [total_all_grns[dataset][method][clone].edges for clone in clones]

        # Create a mapping from each edge to the groups it belongs to
        edge_to_groups = {}
        for group_name, edge_set in zip(clones, edge_sets):
            for edge in edge_set:
                edge_to_groups.setdefault(edge, set()).add(group_name)

        # Convert to list of group memberships
        memberships = [tuple(sorted(group_list)) for group_list in edge_to_groups.values()]

        clone_sum = 0
        norm_sum = 0
        count_clone = 0
        count_norm = 0
        print(dict(Counter(memberships)))
        for key,val in dict(Counter(memberships)).items():
            if len(key)>1 and len(key)<len(clones) and 'N' in key:
                count_norm+=1
                norm_sum+=val
            elif len(key)>1 and len(key)<len(clones) and 'N' not in key:
                count_clone+=1
                clone_sum+=val


        if clone_sum!=0 or count_clone!=0:
            method_setvals['clone v clone'].append(clone_sum/count_clone)
            method_specific_setvals[method]['clone v clone'].append(clone_sum/count_clone)
            dataset_specific_setvals[dataset]['clone v clone'].append(clone_sum/count_clone)
        else:
            method_setvals['clone v clone'].append(0)
            method_specific_setvals[method]['clone v clone'].append(0)
            dataset_specific_setvals[dataset]['clone v clone'].append(0)
        if norm_sum!=0 or count_norm!=0:
            method_setvals['normal v clone'].append(norm_sum/count_norm)
            method_specific_setvals[method]['normal v clone'].append(norm_sum/count_norm)
            dataset_specific_setvals[dataset]['normal v clone'].append(norm_sum/count_norm)
        else:
            method_setvals['normal v clone'].append(0)
            method_specific_setvals[method]['normal v clone'].append(0)
            dataset_specific_setvals[dataset]['normal v clone'].append(0)
        print(clone_sum, norm_sum)
        # Plot
        #fig = plt.figure(figsize=(12, 6))
        #UpSet(data, subset_size='count', show_counts=True).plot(fig=fig)

        #plt.suptitle("GRN Edge Overlaps", fontsize=16)
        #plt.tight_layout()
        #fig.savefig(f"{SAMPLE_PATH}/upset-edges-{SAMPLE}-dpi{DPI}.png", dpi=DPI)
        #plt.show()

In [ ]:
from itertools import combinations

diffsn = []
total_lengthsn = []
diffsc = []
total_lengthsc = []

for sample in total_all_grns.keys():
    clone_groups = total_all_grns[sample]['celloracle'].keys()
    for clone in clone_groups:
        total_lengthsn.append(len(total_all_grns[sample]['celloracle'][clone].edges))
        if clone!='N':
            total_lengthsc.append(len(total_all_grns[sample]['celloracle'][clone].edges))
    for g1, g2 in combinations(clone_groups, 2):
        if g1=='N' or g2=='N':
            diffsn.append(len(total_all_grns[sample]['celloracle'][g1].edges ^ total_all_grns[sample]['celloracle'][g2].edges))
        else:
            diffsc.append(len(total_all_grns[sample]['celloracle'][g1].edges ^ total_all_grns[sample]['celloracle'][g2].edges))

print(np.mean(diffsc)/np.mean(total_lengthsc), np.mean(diffsn)/np.mean(total_lengthsn))

In [ ]:
### JC Calculations instead ###

from itertools import combinations

method_setvals = {'clone v clone': [], 'normal v clone': []}
method_specific_setvals = {
    method: {'clone v clone': [], 'normal v clone': []}
    for method in methods
}
dataset_specific_setvals = {
    dataset: {'clone v clone': [], 'normal v clone': []}
    for dataset in consensus_all_grns.keys()
}

for dataset in consensus_all_grns.keys():
    print(dataset)
    
    for method in methods:
        groups = list(total_all_grns[dataset][method].keys())
        
        # Separate clones and normals
        clone_groups = [g for g in groups if 'N' not in g]
        normal_groups = [g for g in groups if 'N' in g]

        # Get edge sets
        edge_sets = {
            g: set(total_all_grns[dataset][method][g].edges)
            for g in groups
        }

        # --------------------------
        # Clone vs Clone Jaccard
        # --------------------------
        clone_jaccards = []
        for g1, g2 in combinations(clone_groups, 2):
            A = edge_sets[g1]
            B = edge_sets[g2]
            union = A | B
            if len(union) > 0:
                jacc = len(A & B) / len(union)
                clone_jaccards.append(jacc)

        avg_clone_jacc = (
            sum(clone_jaccards) / len(clone_jaccards)
            if clone_jaccards else 0
        )

        # --------------------------
        # Clone vs Normal Jaccard
        # --------------------------
        norm_jaccards = []
        for g1 in clone_groups:
            for g2 in normal_groups:
                A = edge_sets[g1]
                B = edge_sets[g2]
                union = A | B
                if len(union) > 0:
                    jacc = len(A & B) / len(union)
                    norm_jaccards.append(jacc)

        avg_norm_jacc = (
            sum(norm_jaccards) / len(norm_jaccards)
            if norm_jaccards else 0
        )

        # Store results
        method_setvals['clone v clone'].append(avg_clone_jacc)
        method_setvals['normal v clone'].append(avg_norm_jacc)

        method_specific_setvals[method]['clone v clone'].append(avg_clone_jacc)
        method_specific_setvals[method]['normal v clone'].append(avg_norm_jacc)

        dataset_specific_setvals[dataset]['clone v clone'].append(avg_clone_jacc)
        dataset_specific_setvals[dataset]['normal v clone'].append(avg_norm_jacc)

        print("Clone-Clone:", clone_jaccards, 
              "Clone-Normal:", norm_jaccards)

In [ ]:
sns.boxplot(data=pd.DataFrame(combined_setvals).melt(var_name='Group', value_name='No. of common edges'), x='Group', y='No. of common edges', palette='tab10')
plt.title('No. of common edges across clone/normal using consensus GRNs', fontsize=16, pad=20)
plt.xlabel('Group')
plt.ylabel('No. of common edges')
plt.show()
sns.boxplot(data=pd.DataFrame(method_setvals).melt(var_name='Group', value_name='No. of common edges'), x='Group', y='No. of common edges', palette='tab10')
plt.title('No. of common edges across clone/normal using all methods', fontsize=16, pad=20)
plt.xlabel('Group')
plt.ylabel('No. of common edges')
plt.show()
sns.boxplot(data=pd.DataFrame(method_specific_setvals['celloracle']).melt(var_name='Group', value_name='No. of common edges'), x='Group', y='No. of common edges', palette='tab10')
plt.title('No. of common edges across clone/normal using CellOracle', fontsize=16, pad=20)
plt.xlabel('Group')
plt.ylabel('No. of common edges')
plt.show()
sns.boxplot(data=pd.DataFrame(method_specific_setvals['grnboost']).melt(var_name='Group', value_name='No. of common edges'), x='Group', y='No. of common edges', palette='tab10')
plt.title('No. of common edges across clone/normal using GRNBoost', fontsize=16, pad=20)
plt.xlabel('Group')
plt.ylabel('No. of common edges')
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu

sns.set_theme(font_scale=1.5)
sns.set_style("white")
ax = sns.boxplot(data=pd.DataFrame({'clone v clone': [x for k,v in dataset_specific_setvals.items() for x in v['clone v clone'] if ('CID' in k) or ('Triple' in k)], 'normal v clone': [x for k,v in dataset_specific_setvals.items() for x in v['normal v clone'] if ('CID' in k)  or ('Triple' in k)]}).melt(var_name='Group', value_name='No. of common edges'), x='Group', y='No. of common edges', palette=sns.color_palette("tab10")[4:])
ax.tick_params(left=True)
plt.title('Pairwise jaccard similarity across clone/normal', fontsize=16, pad=20)
plt.xlabel('TNBC')
plt.ylabel('Pairwise Jaccard similarity')
plt.show()
stat, p_value = mannwhitneyu([x for k,v in dataset_specific_setvals.items() for x in v['clone v clone'] if ('CID' in k) or ('Triple' in k)], [x for k,v in dataset_specific_setvals.items() for x in v['normal v clone'] if ('CID' in k)  or ('Triple' in k)], alternative='greater')
print(stat,p_value)
ax = sns.boxplot(data=pd.DataFrame({'clone v clone': [x for k,v in dataset_specific_setvals.items() for x in v['clone v clone'] if 'SMC' in k], 'normal v clone': [x for k,v in dataset_specific_setvals.items() for x in v['normal v clone'] if 'SMC' in k]}).melt(var_name='Group', value_name='No. of common edges'), x='Group', y='No. of common edges', palette=sns.color_palette("tab10")[4:])
ax.tick_params(left=True)
plt.title('Pairwise jaccard similarity across clone/normal', fontsize=16, pad=20)
plt.xlabel('CRC')
plt.ylabel('Pairwise Jaccard similarity')
plt.show()
stat, p_value = mannwhitneyu([x for k,v in dataset_specific_setvals.items() for x in v['clone v clone'] if 'SMC' in k], [x for k,v in dataset_specific_setvals.items() for x in v['normal v clone'] if 'SMC' in k], alternative='greater')
print(stat,p_value)

In [ ]:
dataset_specific_setvals

In [ ]:
import networkx as nx
import numpy as np
import itertools
import pandas as pd
from scipy.spatial.distance import hamming
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu


def compute_edge_presence_vectors(sample_dict):
    """
    sample_dict: {'method1':{'cat1':G,...}, 'method2':{'cat1':G,...} }
    Returns:
        edge_vectors[method][category] = binary numpy vector
        edge_index = list of all edges in sample
    """

    # ---- 1) Collect all edges across ALL methods & categories ----
    all_edges = set()
    for method in sample_dict.values():
        for G in method.values():
            all_edges |= set(map(tuple, map(sorted, G.edges())))

    all_edges = sorted(all_edges)  # consistent ordering
    edge_index = {e: i for i, e in enumerate(all_edges)}

    # ---- 2) Build binary vectors ----
    edge_vectors = {}

    for method, categories in sample_dict.items():
        edge_vectors[method] = {}
        for cat, G in categories.items():
            vec = np.zeros(len(all_edges), dtype=int)
            for e in map(tuple, map(sorted, G.edges())):
                vec[edge_index[e]] = 1
            edge_vectors[method][cat] = vec

    return edge_vectors, all_edges


def compute_distances_avg(edge_vectors):
    """
    edge_vectors = {method: {category: binary_vector}}
    Returns:
        intra_all, inter_all:
            lists of averaged intra/inter distances for each method in the sample
    """

    intra_all = []
    inter_all = []

    for method, cats in edge_vectors.items():

        # Sort categories alphabetically (robust for any key names)
        categories = sorted(cats.keys())
        if len(categories) <= 1:
            continue

        # Define N as the last category
        N_cat = categories[-1]
        other_cats = categories[:-1]

        # -------------------------
        # Compute INTRA distances
        # -------------------------
        intra_distances = [
            hamming(cats[c1], cats[c2])
            for c1, c2 in itertools.combinations(other_cats, 2)
        ]

        # Average INTRA distance for this method
        if len(intra_distances) > 0:
            intra_avg = float(np.mean(intra_distances))
            intra_all.append(intra_avg)

        # -------------------------
        # Compute INTER distances
        # -------------------------
        inter_distances = [
            hamming(cats[c], cats[N_cat])
            for c in other_cats
        ]

        # Average INTER distance for this method
        if len(inter_distances) > 0:
            inter_avg = float(np.mean(inter_distances))
            inter_all.append(inter_avg)
        #print(other_cats, intra_distances, inter_distances)

    return intra_all, inter_all

def compute_distances(edge_vectors):
    """
    For a single sample:
    edge_vectors = {method: {category: vec}}
    Returns:
        intra_all, inter_all = combined lists for the sample
    """

    intra_all = []
    inter_all = []

    for method, cats in edge_vectors.items():

        # ---- Sort categories alphabetically (works for any string keys) ----
        categories = sorted(cats.keys())

        if len(categories) <= 1:
            continue

        # ---- Define N = LAST category ----
        N_cat = categories[-1]
        other_cats = categories[:-1]   # everything except N

        # ---- Intra distances (between all pairs not involving N) ----
        for c1, c2 in itertools.combinations(other_cats, 2):
            d = hamming(cats[c1], cats[c2])
            intra_all.append(d)

        # ---- Inter distances (each other category vs N) ----
        for c in other_cats:
            d = hamming(cats[c], cats[N_cat])
            inter_all.append(d)

    return intra_all, inter_all

def build_intra_inter_for_all_samples(data):
    """
    data: dict of samples → methods → categories → nx graphs
    Returns:
        intra_dict[sample], inter_dict[sample]
    """

    intra_dict = {}
    inter_dict = {}

    for sample, sample_dict in data.items():
        edge_vectors, _ = compute_edge_presence_vectors(sample_dict)
        intra, inter = compute_distances(edge_vectors)

        intra_dict[sample] = intra
        inter_dict[sample] = inter

    return intra_dict, inter_dict



def plot_split_violin(intra_dict, inter_dict, title='Inter vs Intra Category Hamming Distances per Sample'):
    """
    Create split violin plot of intra vs inter distances.
    """

    colors = ['#AA5042', '#D5A021', '#1098F7', '#061A40', '#631A86', '#226F54', '#CE4865', '#C6AD94', '#E3D7FF']
    sample_colors = {}
    
    # Build a dataframe suitable for seaborn
    rows = []
    color_index = 0
    for sample in intra_dict:
        sample_colors[sample] = {'intra':colors[color_index],'inter':colors[color_index]}
        color_index+=1
        for v in intra_dict[sample]:
            rows.append({"sample": sample, "distance": v, "type": "intra", "pos":0})
        for v in inter_dict[sample]:
            rows.append({"sample": sample, "distance": v, "type": "inter", "pos":0})

    df = pd.DataFrame(rows)
    
    #for val in df['sample'].unique():
    group1 = df[(df['type'] == 'intra')]['distance']#df[(df['sample'] == val) & (df['type'] == 'intra')]['distance']
    group2 = df[(df['type'] == 'inter')]['distance']#df[(df['sample'] == val) & (df['type'] == 'inter')]['distance']
    #print(group1, group2)
    stat, p_value = mannwhitneyu(group1, group2)
    print(title, stat, f"{p_value:e}")

    plt.figure(figsize=(9, 6))
    vp = sns.violinplot(
            x="pos",
            y="distance",
            hue="type",
            data=df,
            split=True,
            inner='quart',
            fill=False,
            palette={'intra':'#9372B2','inter':'#845B53'},
            legend=False
        )
    vp.tick_params(left=True)
    for sample in intra_dict.keys():
        #sns.violinplot(
        #    x="pos",
        #    y="distance",
        #    hue="type",
        #    data=df.loc[df['sample']==sample],
        #    split=True,
        #    inner="quartile",
        #    fill=False,
        #    palette=sample_colors[sample]
        #)
        temp_df = df.loc[df['sample']==sample].groupby('type', as_index=False)['distance'].mean()
        temp_df['sample'] = sample
        temp_df['pos'] = 0
        temp_df = temp_df.reindex([1,0])
        #print(temp_df)
        ax = sns.boxplot(
            x="pos",
            y="distance",
            hue="type",
            data=temp_df,
            showfliers=False,
            showbox=False,
            width=0.05,
            linecolor=sample_colors[sample]['intra'],
            linewidth=2,
            legend=False,
            medianprops=dict(label="median_star")
        )
        for line in ax.get_lines():
            if line.get_label() == 'median_star':
                # Change the line to a marker-only style
                line.set_linestyle('')
                line.set_marker('*')

    legend_handles = [mpatches.Patch(color=color['intra'], label=label.replace('Triple_negative_BRCA1_0131', 'TN0131')) for label, color in sample_colors.items() if label in ['CID4495', 'CID4513', 'CID44971', 'Triple_negative_BRCA1_0131', 'SMC16', 'SMC07']]
    plt.legend(handles=legend_handles, title='Samples', bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.title(title)
    plt.ylabel('Hamming distance')
    plt.xlabel('')
    plt.tight_layout()
    plt.show()
    
    return vp

'''
temp_dict = total_all_grns.copy()
for dataset in consensus_all_grns.keys():
    #temp_dict[dataset]['consensus'] = consensus_all_grns[dataset].copy()
    for method in ['celloracle']:#['celloracle','grnboost','consensus']:#
        for key in temp_dict[dataset]['celloracle'].keys():
            temp_dict[dataset][method][key].remove_nodes_from(list(nx.isolates(temp_dict[dataset][method][key])))

intra_dict, inter_dict = build_intra_inter_for_all_samples(temp_dict)
'''
#plot_split_violin(intra_dict, inter_dict)

In [ ]:
temp_dict = {dataset:{'celloracle':total_all_grns[dataset]['celloracle']} for dataset in total_all_grns.keys()}
for dataset in total_all_grns.keys():
    for key in total_all_grns[dataset]['celloracle'].keys():
        temp_dict[dataset]['celloracle'][key].remove_nodes_from(list(nx.isolates(temp_dict[dataset]['celloracle'][key])))

intra_dict, inter_dict = build_intra_inter_for_all_samples(temp_dict)
#plot_split_violin(intra_dict, inter_dict, 'Inter vs Intra Category Hamming Distances per Sample (CellOracle)')

In [ ]:
sns.set_theme(font_scale=1.8)
sns.set_style('white')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if ('CID' in key) or ('Triple' in key)]}, {'all samples': [x for key, value in inter_dict.items() for x in value if ('CID' in key) or ('Triple' in key)]}, 'Hamming Distances between N/T across all samples TNBC (CellOracle)')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if 'SM' in key]}, {'all samples': [x for key, value in inter_dict.items() for x in value if 'SM' in key]}, 'Hamming Distances between N/T across all samples CRC (CellOracle)')
vp1 = plot_split_violin({key:value for key, value in intra_dict.items() if ('CID' in key) or ('Triple' in key)}, {key:value for key, value in inter_dict.items() if ('CID' in key) or ('Triple' in key)}, 'TNBC')
vp2 = plot_split_violin({key:value for key, value in intra_dict.items() if 'SM' in key}, {key:value for key, value in inter_dict.items() if 'SM' in key}, 'CRC')
#plot_split_violin({'all samples': [x for key, value in intra_dict.items() for x in value if 'Triple' in key]}, {'all samples': [x for key, value in inter_dict.items() for x in value if 'Triple' in key]}, 'Inter vs Intra Category Hamming Distances all samples TNBC (CellOracle)')

In [ ]:
print(intra_dict)
print(inter_dict)

In [ ]:
dir(vp1)

In [ ]:
vp1.hist()

In [ ]:
[sample for sample in intra_dict]

In [ ]:
temp_dict = {dataset:{'grnboost':total_all_grns[dataset]['grnboost']} for dataset in total_all_grns.keys()}
for dataset in total_all_grns.keys():
    temp_dict[dataset]['grnboost'][key].remove_nodes_from(list(nx.isolates(temp_dict[dataset]['grnboost'][key])))

intra_dict, inter_dict = build_intra_inter_for_all_samples(temp_dict)
plot_split_violin(intra_dict, inter_dict, 'Inter vs Intra Category Hamming Distances per Sample (GRNBoost)')

In [ ]:
temp_dict = {dataset:{'consensus':consensus_all_grns[dataset].copy()} for dataset in total_all_grns.keys()}
for dataset in total_all_grns.keys():
    temp_dict[dataset]['consensus'][key].remove_nodes_from(list(nx.isolates(temp_dict[dataset]['consensus'][key])))

intra_dict, inter_dict = build_intra_inter_for_all_samples(temp_dict)
plot_split_violin(intra_dict, inter_dict, 'Inter vs Intra Category Hamming Distances per Sample (Consensus)')

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

datasets = ['/BC/CID4513', '/BC/CID4515', '/BC/CID4523', '/BC/CID44971', '/BC/CID44991', '/CRC/SMC01', '/CRC/SMC04', '/CRC/SMC07', '/CRC/SMC14', '/CRC/SMC18', '/CRC/SMC23', '/CRC/SMC25']

total_G_df = {}
total_avg_G_df = {}
total_all_grns = {}
total_all_grns_combined = {}

for dataset in tqdm(datasets):
    methods=['grnboost','celloracle']

    SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+dataset+"_frac0.9" for method in methods}

    SAMPLE = SAMPLE_PATHS['celloracle'].split('/')[-1].split('_')[0]

    clones = get_clones(SAMPLE_PATHS['celloracle'])

    temp = []
    for clone in clones:
        if clone.isdigit() or clone=='N':
            temp.append(clone)

    clones = temp.copy()

    print(clones)

    boot_grns = {x:{} for x in methods}

    for method in methods:
        boot_grns[method] = {x:{} for x in clones}
        for clone in clones:
            for rate in [0.9]:
                boot_grns[method][clone][rate] = []
                for i in range(100):
                    network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-sub{i}-"+method+"-net.tsv", sep='\t')
                    if method=='celloracle':
                        G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                    if method=='grnboost':
                        thresh = np.histogram(network['importance'])[1][1]
                        G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                    boot_grns[method][clone][rate].append(G)

    all_grns = {x:{} for x in methods}

    for method in methods:
        for clone in clones:
            network = pd.read_csv(f"{SAMPLE_PATHS[method].replace('0.9','1.0')}/CL{clone}-sub0-"+method+"-net.tsv", sep='\t')
            if method=='celloracle':
                G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
            if method=='grnboost':
                thresh = np.histogram(network['importance'])[1][1]
                G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
            all_grns[method][clone] = G
    
        
        

# Relation between clone size and GRNs

In [ ]:
#source	target	coef_mean	support_0.9

def compute_wasserstein(G1, G2, weight, s):
    # collect targets from both graphs
    t1 = {v for _, v in G1.out_edges(s)}
    t2 = {v for _, v in G2.out_edges(s)}
    targets = list(t1 | t2)
    
    v1 = []
    v2 = []
    
    for t in targets:
        w1 = G1[s][t][weight] if G1.has_edge(s, t) else 0
        w2 = G2[s][t][weight] if G2.has_edge(s, t) else 0
        
        v1.append(w1)
        v2.append(w2)
    
    v1 = np.array(v1, dtype=float)
    v2 = np.array(v2, dtype=float)
    
    # L1 normalization
    if v1.sum() > 0:
        v1 = v1 / np.sum(np.abs(v1))
    if v2.sum() > 0:
        v2 = v2 / np.sum(np.abs(v2))
    
    return np.sum(np.abs(v1 - v2))

def clone_wasserstein(G1, G2, weight='coef_mean'):
    # source nodes = nodes with outgoing edges
    src1 = {u for u, _ in G1.edges()}
    src2 = {u for u, _ in G2.edges()}
    
    common_sources = src1 & src2
    if len(common_sources) == 0:
        return 0
    
    total = 0
    
    for s in common_sources:
        s_dist = compute_wasserstein(G1, G2, weight, s)
        total += s_dist
    
    return total / len(common_sources)


import networkx as nx

def kth_neighbor_jaccard(G1, G2, node, k=1):
    """
    Compute Jaccard similarity of the k-hop neighbors of a node in two graphs.
    """

    # get nodes within distance k
    n1 = set(nx.single_source_shortest_path_length(G1, node, cutoff=k).keys())
    n2 = set(nx.single_source_shortest_path_length(G2, node, cutoff=k).keys())

    # remove the node itself
    n1.discard(node)
    n2.discard(node)

    union = n1 | n2
    if len(union) == 0:
        return 0

    intersection = n1 & n2

    return len(intersection) / len(union), len(intersection), len(union)

def tf_outlier_scoring(G1, G2, tf, btw_tf_g1, btw_tf_g2):
    delta_c = np.abs(btw_tf_g1 - btw_tf_g2)
    b_v = (np.mean([G1[x[0]][x[1]].get('support_0.9') for x in G1.in_edges(tf)])+np.mean([G1[x[0]][x[1]].get('support_0.9') for x in G2.in_edges(tf)]))/2
    successorsg1 = set(G1.successors(tf))
    successorsg2 = set(G2.successors(tf))
    j_c = len(successorsg1.intersection(successorsg2))/len(union(successorsg1.union(successorsg2)))
    w1 = compute_wasserstein(G1, G2, 'coef_mean', tf)
    
    return 0.25*(1 - j_c) + 0.25*(w1) + 0.25*(delta_c) + 0.25*(1-b_v)

#def significant_tf_outliers(graphs, betweeness):
    

In [ ]:
from scipy.stats import binomtest, wilcoxon, spearmanr
#import pingouin as pg

#Make sure to load in all data total_GRN pkl first

all_data = {'Wu_2021':['CID4495', 'CID4513', 'CID4515', 'CID4523', 'CID44971', 'CID44991'],
            'Lee_2020':['SMC01', 'SMC04', 'SMC07', 'SMC09', 'SMC16', 'SMC18'],
            'Pal_2021':['Triple_negative_0126', 'Triple_negative_BRCA1_0131', 'Triple_negative_BRCA1_0554']}

sizes = {}

D = {}
clone_divergence = {}

for paper, SAMPLES in all_data.items():
    for SAMPLE in SAMPLES:
        sizes[SAMPLE] = {}
        if True:#'SMC' in SAMPLE:
            adata = sc.read_h5ad('/shared/nr57/GRN/data/3CA/'+paper+'/'+SAMPLE+'_clone_correct.h5ad')
            value_counts = adata.obs['inferred_clone'].value_counts().to_dict()
            value_counts = {k:v for k,v in value_counts.items() if k in total_all_grns[SAMPLE]['celloracle'].keys()}
            sizes[SAMPLE]['normal'] = value_counts['N']
            del value_counts['N']
            clone1 = max(value_counts, key=value_counts.get)
            clone2 = min(value_counts, key=value_counts.get)
            sizes[SAMPLE]['large'] = value_counts[clone1]
            sizes[SAMPLE]['small'] = value_counts[clone2]
            W_larger = clone_wasserstein(total_all_grns[SAMPLE]['celloracle'][clone1], total_all_grns[SAMPLE]['celloracle']['N'])
            W_smaller = clone_wasserstein(total_all_grns[SAMPLE]['celloracle'][clone2], total_all_grns[SAMPLE]['celloracle']['N'])
            D[SAMPLE] = W_larger - W_smaller
            clone_divergence[SAMPLE] = {}
            clone_divergence[SAMPLE]['larger'] = {'size':value_counts[clone1], 'W':W_larger, 'size_norm':value_counts[clone1]/len(adata.obs)}
            clone_divergence[SAMPLE]['smaller'] = {'size':value_counts[clone2], 'W':W_smaller, 'size_norm':value_counts[clone1]/len(adata.obs)}

T = sum([1 if x > 0 else 0 for x in D.values()])
binom_result = binomtest(T, n=15, p=0.5, alternative='greater')
wilcoxon_result = wilcoxon([x for x in D.values()], alternative='greater')
spearman_val = spearmanr([x['size'] for y in clone_divergence.values() for x in y.values()], [x['W'] for y in clone_divergence.values() for x in y.values()])
print('Binom test:', binom_result.pvalue, '; Wilcoxon test:', wilcoxon_result.pvalue, '; Spearmann:', spearman_val)

In [ ]:
df = pd.DataFrame(sizes).T.div(pd.DataFrame(sizes).T.sum(axis=1), axis=0)
df.index = [x.replace('BRCA1_','') for x in df.index]
df.index = [x.replace('Triple_negative_','TN') for x in df.index]
ax = df.plot.bar(stacked=True)
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.ylabel("Proportion of cells")
plt.xlabel("Samples")
plt.show()
df = pd.DataFrame(sizes).T
df.index = [x.replace('BRCA1_','') for x in df.index]
df.index = [x.replace('Triple_negative_','TN') for x in df.index]
ax = df.plot.bar()
ax.locator_params(axis='y', nbins=20)
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.ylabel("Cell count")
plt.xlabel("Samples")
plt.show()

In [ ]:
print([x['size'] for y in clone_divergence.values() for x in y.values()])
print([x['W'] for y in clone_divergence.values() for x in y.values()])
print([y for y in clone_divergence.values()])
large_vs_small_df = {'large clone':[x['larger']['W'] for x in clone_divergence.values()], 'small clone':[x['smaller']['W'] for x in clone_divergence.values()]}

ax = sns.boxplot(large_vs_small_df, palette=sns.color_palette("husl", 8)[1:])
ax.tick_params(left=True)
#plt.title('Difference in Wasserstein distances')
plt.ylabel("Wasserstein distance")
plt.xlabel("Size of clone")

# Longitudinal regulatory dynamics

In [ ]:
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

all_data = {'BC_longitude':['pre', 'pre_tumor', 'post1','post1_tumor', 'post2','post2_tumor']}

total_G_df = {}
total_all_grns = {}

for paper, datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['celloracle']

        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/bootstrap_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = dataset
        print('sample',SAMPLE)

        clones = get_clones(SAMPLE_PATHS['celloracle'])

        '''
        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()
        '''
        print('clones',clones)

        boot_grns = {x:{} for x in methods}

        for method in methods:
            boot_grns[method] = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[method][clone][rate] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                        if method=='celloracle':
                            G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                        if method=='grnboost':
                            thresh = np.histogram(network['importance'])[1][1]
                            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        boot_grns[method][clone][rate].append(G)

        all_grns = {x:{} for x in methods}

        for method in methods:
            for clone in clones:
                network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-whole-"+method+"-net.tsv", sep='\t')
                if method=='celloracle':
                    G = nx.from_pandas_edgelist(network.loc[network.p<0.05], source='source', target='target', edge_attr='coef_mean', create_using=nx.DiGraph)
                if method=='grnboost':
                    thresh = np.histogram(network['importance'])[1][1]
                    G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                all_grns[method][clone] = G
                
        for method in methods:
            for clone in clones:
                for rate in [0.90]:
                    for idx in range(100):
                        _G = boot_grns[method][clone][rate][idx]
                        for e in _G.edges():
                            if all_grns[method][clone].has_edge(e[0], e[1]):
                                if not f"support_{rate}" in all_grns[method][clone].edges[e[0], e[1]].keys():
                                    all_grns[method][clone].edges[e[0], e[1]][f"support_{rate}"] = 1
                                else:
                                    all_grns[method][clone].edges[e[0], e[1]][f"support_{rate}"] += 1

        G_data = {method:{clone:{"source": [], "target": [], "support_0.90": []} for clone in clones} for method in methods}
        for method in methods:
            for clone in clones:
                for s, t, vals in all_grns[method][clone].edges(data=True):
                    G_data[method][clone]["source"].append(s)
                    G_data[method][clone]["target"].append(t)
                    try:
                        G_data[method][clone]["support_0.90"].append(vals["support_0.9"])
                    except KeyError:
                        G_data[method][clone]["support_0.90"].append(0)

        G_df = {method:{clone:pd.DataFrame(data=G_data[method][clone]) for clone in clones} for method in methods}

        for method in methods:
            cl_edges = {}
            for clone in clones:
                unique_edges = set(all_grns[method][clone].edges())
                for clone2 in clones:
                    if clone2 != clone:
                        unique_edges = unique_edges - set(all_grns[method][clone2].edges())
                cl_edges[clone] = unique_edges
                G_df[method][clone]["edge"] = list(zip(G_df[method][clone]["source"], G_df[method][clone]["target"]))
                G_df[method][clone]['unique'] = [True if edge in cl_edges[clone] else False for edge in G_df[method][clone].edge]

        total_G_df[dataset] = G_df.copy()
        total_all_grns[dataset] = all_grns.copy()

pkl.dump(total_G_df, open('/shared/nr57/GRN/data/3CA/BC_longitude/total_G_df_new.pkl','wb'))
pkl.dump(total_all_grns, open('/shared/nr57/GRN/data/3CA/BC_longitude/total_all_grns_new.pkl','wb'))

In [ ]:
total_G_df = pkl.load(open('/shared/nr57/GRN/data/3CA/BC_longitude/total_G_df_new.pkl','rb'))
total_all_grns = pkl.load(open('/shared/nr57/GRN/data/3CA/BC_longitude/total_all_grns_new.pkl','rb'))
shuffle_grns = pkl.load(open('/shared/nr57/GRN/data/3CA/BC_longitude/shuffle_GRNs.pkl','rb'))

In [ ]:
W_pre_norm = clone_wasserstein(total_all_grns['pre_tumor']['celloracle']['T'], total_all_grns['pre_tumor']['celloracle']['N'])
W_post1_norm = clone_wasserstein(total_all_grns['post1_tumor']['celloracle']['T'], total_all_grns['post1_tumor']['celloracle']['N'])
W_post2_norm = clone_wasserstein(total_all_grns['post2_tumor']['celloracle']['T'], total_all_grns['post2_tumor']['celloracle']['N'])
print('W(pre,normal):', W_pre_norm,'W(post1,normal):', W_post1_norm,'W(post2,normal):', W_post2_norm)

W_dict = {}
for time_point1 in ['pre_tumor', 'post1_tumor', 'post2_tumor']:
    W_dict[time_point1] = {}
    for time_point2 in ['pre_tumor', 'post1_tumor', 'post2_tumor']:
        W_dict[time_point1][time_point2] = clone_wasserstein(total_all_grns[time_point1]['celloracle']['T'], total_all_grns[time_point2]['celloracle']['T'])

print(pd.DataFrame(W_dict))
sns.heatmap(pd.DataFrame(W_dict), cmap='RdBu')
plt.show()
pre_post1 = W_dict['pre_tumor']['post1_tumor']
pre_post2 = W_dict['pre_tumor']['post2_tumor']

'''
W_dict_clone = {}
for time_point in ['pre', 'post1', 'post2']:
    W_dict_clone[time_point] = {}
    for clone1 in total_all_grns[time_point]['celloracle'].keys():
        W_dict_clone[time_point][clone1] = {}
        for clone2 in total_all_grns[time_point]['celloracle'].keys():
            W_dict_clone[time_point][clone1][clone2] = clone_wasserstein(total_all_grns[time_point]['celloracle'][clone1], total_all_grns[time_point]['celloracle'][clone2])

for time_point, data in W_dict_clone.items():
    print(time_point)
    print(pd.DataFrame(data))
'''

W_dict_inter_clone = {}
clonal_divergence = {}
for time_points in [('pre', 'post1'), ('pre', 'post2')]:
    time_point = time_points[0]+'-'+time_points[1]
    clonal_divergence[time_point] = []
    W_dict_inter_clone[time_point] = {}
    for clone1 in total_all_grns[time_points[0]]['celloracle'].keys():
        if clone1 != 'N':
            W_dict_inter_clone[time_point][clone1] = {}
            for clone2 in total_all_grns[time_points[1]]['celloracle'].keys():
                if clone2 != 'N':
                    W_dict_inter_clone[time_point][clone1][clone2] = clone_wasserstein(total_all_grns[time_points[0]]['celloracle'][clone1], total_all_grns[time_points[1]]['celloracle'][clone2])
                    clonal_divergence[time_point].append(W_dict_inter_clone[time_point][clone1][clone2])

for time_point, data in W_dict_inter_clone.items():
    print(time_point)
    print(pd.DataFrame(data))

ax = sns.histplot(clonal_divergence['pre-post1'], stat='proportion', color='#5F898D', alpha=0.6)
ax.tick_params(left=True, bottom=True)
plt.axvline(pre_post1, linestyle="--", linewidth=2, color='#5F898D')
plt.title('Pre - Post1')
plt.show()

ax = sns.histplot(clonal_divergence['pre-post2'], stat='proportion', color='#5F898D', alpha=0.6)
ax.tick_params(left=True, bottom=True)
plt.axvline(pre_post2, linestyle="--", linewidth=2, color='#5F898D')
plt.title('Pre - Post2')
plt.show()

In [ ]:
J_t1_t2 = {}
intersection_lens = []
union_lens = []
for time_point1 in ['pre_tumor']:
    J_t1_t2[time_point1] = {}
    for time_point2 in ['post1_tumor', 'post2_tumor']:
        print(time_point1, time_point2)
        J_t1_t2[time_point1][time_point2] = {}
        G1 = total_all_grns[time_point1]['celloracle']['T']
        G2 = total_all_grns[time_point2]['celloracle']['T']
        src1 = {u for u, _ in G1.edges()}
        src2 = {u for u, _ in G2.edges()}
        common_sources = src1 & src2
        for v in tqdm(common_sources):
            J_t1_t2[time_point1][time_point2][v] = [kth_neighbor_jaccard(G1, G2, v, k=x)[0] for x in range(1,8)]
            intersection_len = [kth_neighbor_jaccard(G1, G2, v, k=x)[1] for x in range(1,7)]
            union_len = [kth_neighbor_jaccard(G1, G2, v, k=x)[2] for x in range(1,7)]
            intersection_lens+=intersection_len
            union_lens+=union_len
            

In [ ]:
[x for x in J_t1_t2['pre_tumor']['post1_tumor'].values()]

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()  # Flatten for easy iteration

for k,ax in enumerate(axes):
    pre_post1 = [x[0]-x[k+1] for x in J_t1_t2['pre_tumor']['post1_tumor'].values()]
    pre_post2 = [x[0]-x[k+1] for x in J_t1_t2['pre_tumor']['post2_tumor'].values()]
    plot_data = {'pre vs post1':pre_post1, 'pre vs post2':pre_post2}
    axx = sns.histplot(plot_data, kde=True, ax=ax, color='skyblue', stat="density", palette='tab10', legend=False)
    axx.tick_params(left=True, bottom=True)
    ax.set_title(f'k = {k+1}')

plt.tight_layout()
plt.show()

#plt.hist([a for x in J_t1_t2['pre_tumor']['post1_tumor'].values() for a in x], bins=50)
#plt.show()

In [ ]:
#store shuffle GRNs
def get_clones(directory_path):
    """
    Returns a list of file names in the specified directory.
    """
    clones = []
    for entry in os.listdir(directory_path):
        full_path = os.path.join(directory_path, entry)
        if os.path.isfile(full_path):
            clones.append(entry.split('-')[0][2])
    return list(set(clones))

all_data = {'BC_longitude':['pre_tumor', 'post1_tumor', 'post2_tumor']}

all_grns = {}

for paper, datasets in all_data.items():
    for dataset in tqdm(datasets):
        methods=['celloracle']

        SAMPLE_PATHS = {method:"/shared/nr57/GRN/data/3CA/shuffle_"+method+'/'+paper+'/'+dataset for method in methods}

        SAMPLE = dataset
        print('sample',SAMPLE)

        clones = get_clones(SAMPLE_PATHS['celloracle'])

        '''
        temp = []
        for clone in clones:
            if clone.isdigit() or clone=='N':
                temp.append(clone)

        clones = temp.copy()
        '''
        print('clones',clones)

        boot_grns = {x:{} for x in methods}

        for method in methods:
            boot_grns[method] = {x:{} for x in clones}
            for clone in clones:
                for rate in [0.9]:
                    boot_grns[method][clone] = []
                    for i in range(1, 101):
                        network = pd.read_csv(f"{SAMPLE_PATHS[method]}/CL{clone}-{i}-"+method+"-net.tsv", sep='\t')
                        if method=='celloracle':
                            network_df = network.loc[network.p<0.05]
                            subset_df1 = pd.merge(network_df, total_G_df[dataset]['celloracle'][clone], on=['source', 'target'], how='left').fillna(0)
                            G = nx.from_pandas_edgelist(subset_df1, source='source', target='target', edge_attr=['coef_mean', 'support_0.90'], create_using=nx.DiGraph)
                        if method=='grnboost':
                            thresh = np.histogram(network['importance'])[1][1]
                            G = nx.from_pandas_edgelist(network.loc[network.importance>=thresh], source='TF', target='target', edge_attr='importance', create_using=nx.DiGraph)
                        
                        boot_grns[method][clone].append(G)

        all_grns[dataset] = boot_grns[method]

pkl.dump(all_grns, open('/shared/nr57/GRN/data/3CA/BC_longitude/shuffle_GRNs.pkl','wb'))

In [ ]:
from statsmodels.stats.multitest import multipletests

betweeness = {}
all_grns = {}
for dataset in ['pre_tumor', 'post1_tumor', 'post2_tumor']:
    all_grns[dataset] = {'shuffles':shuffle_grns[dataset]['T'], 'whole':total_all_grns[dataset]['celloracle']['T']}
    betweeness[dataset] = {'shuffles':[nx.betweenness_centrality(x) for x in shuffle_grns[dataset]['T']], 'whole':nx.betweenness_centrality(total_all_grns[dataset]['celloracle']['T'])}
common_nodes = nx.intersection([])

pkl.dump(betweeness, open('/shared/nr57/GRN/data/3CA/BC_longitude/tumor_betweeness.pkl','wb'))


tf_outlier_scoring(G1, G2, tf, btw_tf_g1, btw_tf_g2)

nx.betweenness_centrality(shuffle_grns['post1_tumor']['T'][0])

shuffle_grns['post1_tumor']['N'][0]

In [ ]:
from scipy.stats import binomtest, wilcoxon, spearmanr
import itertools

all_data = {'BC_longitude':['pre', 'post1', 'post2']}

D = {}
T = {}
binom_result = {}
clone_divergence = {}
wilcoxon_result = {}

for paper, SAMPLES in all_data.items():
    for SAMPLE in SAMPLES:
        adata = sc.read_h5ad('/shared/nr57/GRN/data/3CA/'+paper+'/'+SAMPLE+'_clone.h5ad')
        value_counts = adata.obs['inferred_clone'].value_counts().to_dict()
        value_counts = {k:v for k,v in value_counts.items() if k in total_all_grns[SAMPLE]['celloracle'].keys()}
        del value_counts['N']
        clone_divergence[SAMPLE] = {}
        D[SAMPLE] = {}
        for clone_pair in list(itertools.combinations(value_counts.keys(), 2)):
            if value_counts[clone_pair[0]] > value_counts[clone_pair[1]]:
                clone1 = clone_pair[0]
                clone2 = clone_pair[1]
            else:
                clone1 = clone_pair[1]
                clone2 = clone_pair[0]
            W_larger = clone_wasserstein(total_all_grns[SAMPLE]['celloracle'][clone1], total_all_grns[SAMPLE]['celloracle']['N'])
            W_smaller = clone_wasserstein(total_all_grns[SAMPLE]['celloracle'][clone2], total_all_grns[SAMPLE]['celloracle']['N'])
            D[SAMPLE][clone1+':'+clone2] = W_larger - W_smaller
            clone_divergence[SAMPLE][clone1+':'+clone2] = {}
            clone_divergence[SAMPLE][clone1+':'+clone2]['larger'] = {'size':value_counts[clone1], 'W':W_larger, 'size_norm':value_counts[clone1]/len(adata.obs)}
            clone_divergence[SAMPLE][clone1+':'+clone2]['smaller'] = {'size':value_counts[clone2], 'W':W_smaller, 'size_norm':value_counts[clone1]/len(adata.obs)}
    
        T[SAMPLE] = sum([1 if x > 0 else 0 for x in D[SAMPLE].values()])
        binom_result[SAMPLE] = binomtest(T[SAMPLE], n=len(D[SAMPLE].keys()), p=0.5, alternative='greater')
        wilcoxon_result[SAMPLE] = wilcoxon([x for x in D[SAMPLE].values()], alternative='greater')

In [ ]:
print(binom_result)
print(wilcoxon_result)

# BC Longitude

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import r2_score
from collections import defaultdict


def find_pairwise_row_outliers(df, n_std=2):

    outliers = {}

    # all pairwise row combinations
    for r1, r2 in combinations(df.index, 2):

        X = df.loc[r1].values
        Y = df.loc[r2].values

        # fit straight line Y = mX + c
        m, c = np.polyfit(X, Y, 1)
        Y_pred = m * X + c

        residuals = Y - Y_pred
        std_res = np.std(residuals)

        pair_key = str(r1)+'::'+str(r2)
        outliers[pair_key] = {'R2':r2_score(X, Y), 'outliers':[]}

        for i, col in enumerate(df.columns):
            if abs(residuals[i]) > n_std * std_res:
                outliers[pair_key]['outliers'].append({
                    "column": col,
                    "point": (Y[i], X[i]),
                    "residual": residuals[i]
                })

    return outliers

def find_sample_outlier_nodes(graph_dict, centrality='degree', thresh=2.0):
    """
    Detect nodes with outlier centrality values across group networks within each sample.

    Parameters
    ----------
    graph_dict : dict
        {'sample': {'group': nx_graph}}
    centrality : str
        'degree', 'betweenness', 'closeness', or 'pagerank'
    z_thresh : float
        Z-score threshold to define outliers

    Returns
    -------
    dict
        {'sample': {'group': [outlier_nodes]}}
    """

    centrality_funcs = {
        'degree': nx.degree_centrality,
        'betweenness': nx.betweenness_centrality,
        'closeness': nx.closeness_centrality,
        'pagerank': nx.pagerank
    }

    if centrality not in centrality_funcs:
        raise ValueError("Unsupported centrality metric")

    results = {}
    sample_df = {}
    sample_zscores = {}

    for sample, groups in graph_dict.items():

        # compute centrality for each group
        group_cent = {}
        all_nodes = set()

        for group, G in groups.items():
            cent = centrality_funcs[centrality](G)
            group_cent[group] = cent
            all_nodes.update(cent.keys())

        all_nodes = sorted(all_nodes)

        # build centrality matrix
        data = []
        for group in group_cent:
            row = [group_cent[group].get(n, 0) for n in all_nodes]
            data.append(row)

        df = pd.DataFrame(data, index=groups.keys(), columns=all_nodes)

        # compute z-scores per node across groups
        #zscores = (df - df.mean()) / df.std(ddof=0)
        #zscores = zscores.fillna(0)

        #sample_outliers = {}

        #for group in zscores.index:
        #    out_nodes = zscores.columns[np.abs(zscores.loc[group]) > z_thresh]
        #    sample_outliers[group] = list(out_nodes)

        results[sample] = find_pairwise_row_outliers(df, n_std=thresh)
        sample_df[sample] = df.copy()
        #sample_zscores[sample] = zscores.copy()

    return sample_df, sample_zscores, results


def build_outlier_matrix(result_dict):
    """
    result_dict format:
    dict[sample][pair] = {'R2': n1, 'outliers': [d1, d2, ...]}
    where di = {'column': val1, 'point': val2, 'residual': val3}
    """

    counts = defaultdict(lambda: defaultdict(int))
    all_columns = set()

    for sample, pairs in result_dict.items():
        for pair, pair_data in pairs.items():
            if 'N' in pair:
                continue
            for d in pair_data.get('outliers', []):
                col = d['column']
                counts[sample][col] += 1
                all_columns.add(col)

    all_columns = sorted(all_columns)

    # build dataframe
    df = pd.DataFrame(0, index=result_dict.keys(), columns=all_columns)

    for sample in counts:
        for col, c in counts[sample].items():
            df.loc[sample, col] = c

    return df

In [ ]:
centralities = {}
for centrality in ['degree', 'betweenness']:
    centralities[centrality] = {}
    samples = {'sample':{'pre_tumor':total_all_grns['pre_tumor']['celloracle']['T'], 'post1_tumor':total_all_grns['post1_tumor']['celloracle']['T'], 'post2_tumor':total_all_grns['post2_tumor']['celloracle']['T']}}
    centralities[centrality]['centrality_df'], centralities[centrality]['zscore_df'], centralities[centrality]['outliers'] = find_sample_outlier_nodes(samples, centrality=centrality, thresh=2)
    centralities[centrality]['matrix'] = build_outlier_matrix(centralities[centrality]['outliers'])

In [ ]:
clone_centralities = {}
for centrality in ['degree', 'betweenness']:
    clone_centralities[centrality] = {}
    samples = {'pre':total_all_grns['pre']['celloracle'], 'post1':total_all_grns['post1']['celloracle'], 'post2':total_all_grns['post2']['celloracle']}
    clone_centralities[centrality]['centrality_df'], clone_centralities[centrality]['zscore_df'], clone_centralities[centrality]['outliers'] = find_sample_outlier_nodes(samples, centrality=centrality, thresh=3)
    clone_centralities[centrality]['matrix'] = build_outlier_matrix(clone_centralities[centrality]['outliers'])

In [ ]:
centralities['betweenness']['outliers']['sample']['pre_tumor::post1_tumor']

In [ ]:
cent = 'degree'
samp = 'sample'

temp_df = pd.DataFrame()
temp_df['pre'] = centralities[cent]['centrality_df'][samp].T[['pre_tumor']]
temp_df['post1'] = centralities[cent]['centrality_df'][samp].T[['post1_tumor']]
temp_df['post2'] = centralities[cent]['centrality_df'][samp].T[['post2_tumor']]

for x,y in [('pre','post1'), ('pre','post2'), ('post1','post2')]:
    print(x,y)
    fig, ax = plt.subplots()
    
    conditions = [
    (temp_df[x] == 0) & (temp_df[y] != 0), # Red condition
    (temp_df[x] != 0) & (temp_df[y] == 0)  # Yellow condition
    ]

    choices = ['#DA2B6B', '#FFC107']

    temp_df['color'] = np.select(conditions, choices, default='#1E88E5')
    
    ax.scatter(temp_df[x], temp_df[y], c=temp_df['color'])
    ax.tick_params(left=True, bottom=True)
    #ax.plot([0, 0, 0.004], [0, 0, 0.004], color='red', linestyle='-')
    ax.plot([0, 0, 0.63], [0, 0, 0.63], color='red', linestyle='-')
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(cent)
    #ax.text(0.0032, 0.0007, r"$R^2$="+f"{metrics.r2_score(temp_df[x], temp_df[y]):0.4f}",
    #        fontsize=18, ha='right', va='bottom')
    ax.text(0.57, 0.15, r"$R^2$="+f"{metrics.r2_score(temp_df[x], temp_df[y]):0.4f}",
            fontsize=18, ha='right', va='bottom')

    label_indices = [a['column'] for a in centralities[cent]['outliers']['sample'][x+'_tumor::'+y+'_tumor']['outliers']]
    print(x,y,label_indices)

    for specific_index in []:
        # 4. Get the x, y coordinates and the label text for that specific index
        # We use .loc to access by index label
        if specific_index in temp_df.index and ((temp_df.loc[specific_index, x] == 0 and temp_df.loc[specific_index, y] > 0.001) or (temp_df.loc[specific_index, x] > 0.001 and temp_df.loc[specific_index, y] == 0)):
            print(x,y,specific_index)
            x_val = temp_df.loc[specific_index, x]
            y_val = temp_df.loc[specific_index, y]
            label_text = specific_index

            # 5. Use ax.annotate to place the label on the plot
            ax.annotate(
                label_text,               # The text to display
                (x_val, y_val),           # The point (x, y) to annotate
                textcoords="offset points", # Position the text relative to the point
                ha='center',               # Horizontal alignment
                rotation=45
            )

    # Display the plot
    #plt.show()


In [ ]:
name = 'PLAGL1'
print(total_G_df['post1_tumor']['celloracle']['T'].loc[(total_G_df['post1_tumor']['celloracle']['T']['source']==name) | (total_G_df['post1_tumor']['celloracle']['T']['target']==name)]['support_0.90'].mean())
print(total_G_df['post2_tumor']['celloracle']['T'].loc[(total_G_df['post2_tumor']['celloracle']['T']['source']==name) | (total_G_df['post2_tumor']['celloracle']['T']['target']==name)]['support_0.90'].mean())
print(total_G_df['pre_tumor']['celloracle']['T'].loc[(total_G_df['pre_tumor']['celloracle']['T']['source']==name) | (total_G_df['pre_tumor']['celloracle']['T']['target']==name)]['support_0.90'].mean())

In [ ]:
cent='betweenness'

prevpost1_btw = ['FOXF2', 'GLI2', 'CREM', 'EGR1']#['TEAD2', 'MYC', 'KLF7', 'ATF3', 'IRF1', 'MEF2C', 'ZEB1']
prevpost2_btw = ['FOXF2', 'GLI2', 'CREM', 'EGR1']#['PLAGL1', 'NFE2', 'SP6', 'ATF3', 'KLF7', 'ZEB1', 'MEF2C', 'IRF1']
post1vpost2_btw = ['FOXF2', 'GLI2', 'CREM', 'EGR1']#['PLAGL1', 'ATF3', 'NFE2', 'SP6', 'KLF7', 'MYC', 'KLF5', 'TEAD2']

for key, pair in {'pre-post1':prevpost1_btw, 'pre-post2':prevpost2_btw, 'post1-post2':post1vpost2_btw}.items():
#for key, pair in {'Pre-Post1':prevpost1_deg, 'Pre-Post2':prevpost2_deg, 'Post1-Post2':post1vpost2_deg}.items():
    plot_df = []
    for clone in ['0', '1', '2', '3', '4', '5', '6', '7']:
        maxval0 = clone_centralities[cent]['centrality_df'][key.split('-')[0]].max(axis=1)[clone]
        maxval1 = clone_centralities[cent]['centrality_df'][key.split('-')[1]].max(axis=1)[clone]
        for gene in pair:
            if gene in clone_centralities[cent]['centrality_df'][key.split('-')[0]].columns:
                val0 = clone_centralities[cent]['centrality_df'][key.split('-')[0]][gene][clone]
                val0 = val0/maxval0
            else:
                val0 = -0.1
            if gene in clone_centralities[cent]['centrality_df'][key.split('-')[1]].columns:
                val1 = clone_centralities[cent]['centrality_df'][key.split('-')[1]][gene][clone]
                val1 = val1/maxval1
            else:
                val1 = -0.1
            #print(val0,val1)
            plot_df.append({'clone':clone, 'gene':gene, 'betweeness':val0, 'treatment':key.split('-')[0]})
            plot_df.append({'clone':clone, 'gene':gene, 'betweeness':val1, 'treatment':key.split('-')[1]})
    plot_df = pd.DataFrame(plot_df)
    ax = sns.catplot(data=plot_df, x="betweeness", y="clone", hue="gene", col="treatment", marker="*", s=100, palette='tab10')
    ax.tick_params(left=True, bottom=True)
    ax.refline(x=0, color='red')
    sns.move_legend(ax, "upper left", bbox_to_anchor=(0.9, 1))
    #plt.title(key.replace('-',' vs '))
    #plt.savefig('/shared/nr57/GRN/plots/treatment/'+key+'-degree.png', dpi=300)
    plt.show()
    plt.close()

In [ ]:
not_all_positive = ~(clone_centralities['betweenness']['centrality_df']['post1'] > 0).all(axis=0)
some_positive = (clone_centralities['betweenness']['centrality_df']['post1'] > 0).any(axis=0)

# Combine the conditions using logical AND (&)
mask = not_all_positive & some_positive

# Select the columns using .loc
clone_centralities['betweenness']['centrality_df']['post1'].loc[:, mask]

In [ ]:
clone_centralities['betweenness']['centrality_df']['post1'].columns[(clone_centralities['betweenness']['centrality_df']['post1'].max() > 0.5) & (clone_centralities['betweenness']['centrality_df']['post1'].min() < 0.5)]

In [ ]:
norm_df = clone_centralities['degree']['centrality_df']['post1'].div(clone_centralities['degree']['centrality_df']['post1'].max(axis=1), axis=0)
norm_df.columns[(norm_df.max() > 0.6) & (norm_df.min() < 0.3)]

In [ ]:
norm_df

In [ ]:
print(len(nx.intersection_all([total_all_grns['pre_tumor']['celloracle']['T'], total_all_grns['post1_tumor']['celloracle']['T'], total_all_grns['post2_tumor']['celloracle']['T']]).edges()))
print(len(nx.intersection(total_all_grns['pre_tumor']['celloracle']['T'], total_all_grns['post2_tumor']['celloracle']['T']).edges()))
print(len(nx.intersection(total_all_grns['post1_tumor']['celloracle']['T'], total_all_grns['post2_tumor']['celloracle']['T']).edges()))
print(len(total_all_grns['pre_tumor']['celloracle']['T'].edges()))
print(len(total_all_grns['post1_tumor']['celloracle']['T'].edges()))
print(len(total_all_grns['post2_tumor']['celloracle']['T'].edges()))